In [0]:
%run /Users/avani.sharma2324@gmail.com/1_config

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
DQ

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T_SILVER_TRIPS} (
  trip_key            STRING  NOT NULL COMMENT 'sha2 business key, dedup + merge key',
  vendor_id           INT,
  pickup_ts           TIMESTAMP,
  dropoff_ts          TIMESTAMP,
  pickup_date         DATE,
  pickup_hour         INT,
  passenger_count     INT,
  trip_distance       DOUBLE,
  trip_duration_min   DOUBLE,
  avg_speed_mph       DOUBLE,
  rate_code_id        INT,
  store_and_fwd_flag  STRING,
  pu_location_id      INT,
  do_location_id      INT,
  payment_type_id     INT,
  fare_amount         DOUBLE,
  extra               DOUBLE,
  mta_tax             DOUBLE,
  tip_amount          DOUBLE,
  tolls_amount        DOUBLE,
  improvement_surcharge DOUBLE,
  congestion_surcharge  DOUBLE,
  airport_fee         DOUBLE,
  total_amount        DOUBLE,
  tip_pct             DOUBLE,
  is_airport_trip     BOOLEAN,
  _source_file        STRING,
  _ingested_at        TIMESTAMP,
  _processed_at       TIMESTAMP
)
USING DELTA
COMMENT 'Cleansed, deduplicated yellow taxi trips'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T_SILVER_QUARANTINE} (
  trip_key STRING, pickup_ts TIMESTAMP, dropoff_ts TIMESTAMP, passenger_count INT,
  trip_distance DOUBLE, fare_amount DOUBLE, total_amount DOUBLE,
  pu_location_id INT, do_location_id INT,
  dq_failures ARRAY<STRING>, _source_file STRING, _quarantined_at TIMESTAMP
) USING DELTA
COMMENT 'Rows rejected by silver DQ rules, kept for investigation'
""")

## Transformations + DQ rules

In [0]:
def conform(df):
    """Rename to snake_case, cast, and derive the columns downstream needs."""
    return (df
        .withColumn("pickup_ts",  F.col("tpep_pickup_datetime").cast("timestamp"))
        .withColumn("dropoff_ts", F.col("tpep_dropoff_datetime").cast("timestamp"))
        .withColumn("trip_key", F.sha2(F.concat_ws("||",
            F.coalesce(F.col("VendorID").cast("string"), F.lit("")),
            F.coalesce(F.col("tpep_pickup_datetime").cast("string"), F.lit("")),
            F.coalesce(F.col("tpep_dropoff_datetime").cast("string"), F.lit("")),
            F.coalesce(F.col("PULocationID").cast("string"), F.lit("")),
            F.coalesce(F.col("DOLocationID").cast("string"), F.lit("")),
            F.coalesce(F.col("trip_distance").cast("string"), F.lit("")),
            F.coalesce(F.col("total_amount").cast("string"), F.lit("")),
        ), 256))
        .withColumn("pickup_date", F.to_date("pickup_ts"))
        .withColumn("pickup_hour", F.hour("pickup_ts"))
        .withColumn("trip_duration_min",
            F.round((F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long")) / 60.0, 2))
        .withColumn("avg_speed_mph",
            F.when(F.col("trip_duration_min") > 0,
                   F.round(F.col("trip_distance") / (F.col("trip_duration_min") / 60.0), 2)))
        .withColumn("vendor_id",       F.col("VendorID").cast("int"))
        .withColumn("rate_code_id",    F.coalesce(F.col("RatecodeID").cast("int"), F.lit(99)))
        .withColumn("payment_type_id", F.coalesce(F.col("payment_type").cast("int"), F.lit(5)))
        .withColumn("pu_location_id",  F.col("PULocationID").cast("int"))
        .withColumn("do_location_id",  F.col("DOLocationID").cast("int"))
        .withColumn("passenger_count", F.col("passenger_count").cast("int"))
        .withColumn("store_and_fwd_flag", F.upper(F.trim(F.coalesce(F.col("store_and_fwd_flag"), F.lit("N")))))
        .withColumn("airport_fee",          F.coalesce(F.col("airport_fee").cast("double"), F.lit(0.0)))
        .withColumn("congestion_surcharge", F.coalesce(F.col("congestion_surcharge").cast("double"), F.lit(0.0)))
        .withColumn("tip_pct",
            F.when(F.col("fare_amount") > 0,
                   F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 2)).otherwise(F.lit(0.0)))
        .withColumn("is_airport_trip",
            F.col("PULocationID").isin(1, 132, 138) | F.col("DOLocationID").isin(1, 132, 138))
    )


def add_dq_failures(df):
    """One array column listing every rule a row broke. Empty array = clean."""
    rules = [
        (F.col("pickup_ts").isNull() | F.col("dropoff_ts").isNull(),            "null_timestamp"),
        (F.col("pu_location_id").isNull() | F.col("do_location_id").isNull(),   "null_location"),
        (F.col("dropoff_ts") <= F.col("pickup_ts"),                             "dropoff_before_pickup"),
        (F.col("trip_duration_min") < DQ["min_duration_min"],                   "duration_too_short"),
        (F.col("trip_duration_min") > DQ["max_duration_min"],                   "duration_too_long"),
        (F.col("trip_distance") < DQ["min_trip_distance"],                      "distance_too_short"),
        (F.col("trip_distance") > DQ["max_trip_distance"],                      "distance_too_long"),
        (F.col("passenger_count").isNull() | (F.col("passenger_count") < 1),    "invalid_passenger_count"),
        (F.col("passenger_count") > DQ["max_passengers"],                       "too_many_passengers"),
        (F.col("fare_amount") < 0,                                              "negative_fare"),
        (F.col("total_amount") <= DQ["min_total_amount"],                       "non_positive_total"),
        (F.col("total_amount") > DQ["max_total_amount"],                        "total_out_of_range"),
        (F.col("pickup_date") < F.lit(DQ["valid_from_date"]).cast("date"),      "pickup_date_too_old"),
        (F.col("pickup_date") > F.lit(DQ["valid_to_date"]).cast("date"),        "pickup_date_in_future"),
        (F.col("avg_speed_mph") > 90,                                           "implausible_speed"),
    ]
    arr = F.array(*[F.when(cond, F.lit(name)) for cond, name in rules])
    return df.withColumn("dq_failures", F.array_compact(arr))


SILVER_COLS = ["trip_key","vendor_id","pickup_ts","dropoff_ts","pickup_date","pickup_hour",
    "passenger_count","trip_distance","trip_duration_min","avg_speed_mph","rate_code_id",
    "store_and_fwd_flag","pu_location_id","do_location_id","payment_type_id","fare_amount",
    "extra","mta_tax","tip_amount","tolls_amount","improvement_surcharge","congestion_surcharge",
    "airport_fee","total_amount","tip_pct","is_airport_trip","_source_file","_ingested_at"]

## foreachBatch: quarantine + MERGE

In [0]:
def process_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    # Bronze ka _ingest_time column Silver mein _ingested_at ban jata hai
    batch_df = batch_df.withColumn("_ingested_at", F.col("_ingest_time"))

    scored = add_dq_failures(conform(batch_df))

    # --- rejects -> quarantine ---
    (scored.filter(F.size("dq_failures") > 0)
        .select("trip_key","pickup_ts","dropoff_ts","passenger_count","trip_distance",
                "fare_amount","total_amount","pu_location_id","do_location_id",
                "dq_failures","_source_file",
                F.current_timestamp().alias("_quarantined_at"))
        .write.mode("append").saveAsTable(T_SILVER_QUARANTINE))

    # --- clean rows: batch ke andar dedup (latest ingest wins), phir MERGE ---
    w = Window.partitionBy("trip_key").orderBy(F.col("_ingested_at").desc())
    clean = (scored.filter(F.size("dq_failures") == 0)
        .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
        .select(*SILVER_COLS)
        .withColumn("_processed_at", F.current_timestamp()))

    (DeltaTable.forName(spark, T_SILVER_TRIPS).alias("t")
        .merge(clean.alias("s"), "t.trip_key = s.trip_key")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

In [0]:
q = (spark.readStream
        .table(T_BRONZE_TRIPS)
        .writeStream
        .foreachBatch(process_batch)
        .option("checkpointLocation", CKPT)
        .trigger(availableNow=True)
        .start())
q.awaitTermination()
print("silver load finished")

## Zone staging 

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {T_SILVER_ZONE_STG} AS
SELECT
  CAST(LocationID AS INT)                       AS location_id,
  initcap(trim(Borough))                        AS borough,
  trim(Zone)                                    AS zone_name,
  trim(service_zone)                            AS service_zone,
  sha2(concat_ws('||', trim(Borough), trim(Zone), trim(service_zone)), 256) AS hash_diff,
  current_timestamp()                           AS _staged_at
FROM {T_BRONZE_ZONES}
WHERE LocationID IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY LocationID ORDER BY _ingest_time DESC) = 1
""")

display(spark.sql(f"SELECT count(*) AS zones FROM {T_SILVER_ZONE_STG}"))

## Verify

In [0]:
display(spark.sql(f"""
SELECT (SELECT count(*) FROM {T_BRONZE_TRIPS})      AS bronze_rows,
       (SELECT count(*) FROM {T_SILVER_TRIPS})      AS silver_rows,
       (SELECT count(*) FROM {T_SILVER_QUARANTINE}) AS quarantined_rows
"""))

In [0]:
display(spark.sql(f"""
SELECT reason, count(*) AS rows_rejected
FROM (SELECT explode(dq_failures) AS reason FROM {T_SILVER_QUARANTINE})
GROUP BY reason ORDER BY rows_rejected DESC
"""))

In [0]:
display(spark.table(T_SILVER_TRIPS).limit(10))

In [0]:
import pyspark.sql.functions as F

silver_stream = (
    spark.readStream
    .table("nyc_taxi.bronze.trips_raw")
    .withWatermark("tpep_pickup_datetime", "3 days")  
    .filter(F.col("tpep_pickup_datetime").isNotNull())
)